<a href="https://colab.research.google.com/github/andrewdigiacomo27/econ8310-assignment4/blob/main/assignment_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4
## Econ 8310 - Business Forecasting

This assignment will make use of the bayesian statistical models covered in Lessons 10 to 12.

A/B Testing is a critical concept in data science, and for many companies one of the most relevant applications of data-driven decision-making. In order to improve product offerings, marketing campaigns, user interfaces, and many other user-facing interactions, scientists and engineers create experiments to determine the efficacy of proposed changes. Users are then randomly assigned to either the treatment or control group, and their behavior is recorded.
If the changes that the treatment group is exposed to can be measured to have a benefit in the metric of interest, then those changes are scaled up and rolled out to across all interactions.
Below is a short video detailing the A/B Testing process, in case you want to learn a bit more:
[https://youtu.be/DUNk4GPZ9bw](https://youtu.be/DUNk4GPZ9bw)

For this assignment, you will use an A/B test data set, which was pulled from the Kaggle website (https://www.kaggle.com/datasets/yufengsui/mobile-games-ab-testing). I have added the data from the page into Codio for you. It can be found in the cookie_cats.csv file in the file tree. It can also be found at [https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv](https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv)

The variables are defined as follows:

| Variable Name  | Definition |
|----------------|----|
| userid         | A unique number that identifies each player  |
| version        | Whether the player was put in the control group (gate_30 - a gate at level 30) or the group with the moved gate (gate_40 - a gate at level 40) |
| sum_gamerounds | The number of game rounds played by the player during the first 14 days after install.  |
| retention1     | Did the player come back and play 1 day after installing?     |
| retention7     | Did the player come back and play 7 days after installing?    |               

### The questions

You will be asked to answer the following questions in a small quiz on Canvas:
1. What was the effect of moving the gate from level 30 to level 40 on 1-day retention rates?
2. What was the effect of moving the gate from level 30 to level 40 on 7-day retention rates?
3. What was the biggest challenge for you in completing this assignment?

You will also be asked to submit a URL to your forked GitHub repository containing your code used to answer these questions.

In [8]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az

In [9]:
data = pd.read_csv("https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv")
print(data)

        userid  version  sum_gamerounds  retention_1  retention_7
0          116  gate_30               3        False        False
1          337  gate_30              38         True        False
2          377  gate_40             165         True        False
3          483  gate_40               1        False        False
4          488  gate_40             179         True         True
...        ...      ...             ...          ...          ...
90184  9999441  gate_40              97         True        False
90185  9999479  gate_40              30        False        False
90186  9999710  gate_30              28         True        False
90187  9999768  gate_40              51         True        False
90188  9999861  gate_40              16        False        False

[90189 rows x 5 columns]


In [10]:
# data['gate_30'] = (data['version'] == 'gate_40').astype(int)
# coords = {"observations": data.index.values}

In [11]:
# coords = {"observations": data.index.values}

# with pm.Model(coords = coords) as gate_retention_model:
#   n = data.shape[0]
#   gate30 = pm.Data('gate30', data['gate30'], dims = "observations")
#   r1 = pm.Data('retention_1', data['retention_1'], dims = "observations")
#   sum_gamerounds = pm.Data('sum_gamerounds', data['sum_gamerounds'], dims = "oberservations")

#   #priors
#   B0 = pm.Normal("B0", mu=0, sigma=1)
#   B_gate30 = pm.Normal("B_gate30", mu=0, sigma=1)
#   B_r1 = pm.Normal("B_r1", mu=0, sigma=1)
#   B_r7 = pm.Normal("B_r7", mu=0, sigma=1)
#   b_gamerounds = pm.Normal("B_gamerounds", mu=0, sigma=1)

#   #linear model
#   u = B0
#   sigma = pm.halfNormal("sigma", sigma=5, testval=1.0)

#   likelihood = pm.Normal('y', mu=u, sigma=sigma, observed=data)


In [12]:
# coords = {"observations": data.index.values}

# with pm.Model(coords = coords) as gate_retention_model:
#   n = data.shape[0]
#   gate30 = pm.Data('gate30', data['gate30'], dims = "observations")
#   r7 = pm.Data('retention_7', data['retention_7'], dims = "observations")

#   #priors
#   B0 = pm.Normal("B0", mu=0, sigma=1)
#   B_gate30 = pm.Normal("B_gate30", mu=0, sigma=1)


#   #linear model
#   u = B0 + gate30 + B_gate30 + gate40 + B_gate40 + r1 + B_r1 + r7 + B_r7
#   sigma = pm.halfNormal("sigma", sigma=5, testval=1.0)

#   likelihood = pm.Normal('y', mu=u, sigma=sigma, observed=data)

In [13]:
data['retention_1'] = data['retention_1'].astype(int)
data['retention_7'] = data['retention_7'].astype(int)
gate30 = data[data['version'] == 'gate_30']
gate40 = data[data['version'] == 'gate_40']
print(gate30)
print(gate40)

        userid  version  sum_gamerounds  retention_1  retention_7
0          116  gate_30               3            0            0
1          337  gate_30              38            1            0
6         1066  gate_30               0            0            0
11        2101  gate_30               0            0            0
13        2179  gate_30              39            1            0
...        ...      ...             ...          ...          ...
90179  9998576  gate_30              14            1            0
90180  9998623  gate_30               7            0            0
90182  9999178  gate_30              21            1            0
90183  9999349  gate_30              10            0            0
90186  9999710  gate_30              28            1            0

[44700 rows x 5 columns]
        userid  version  sum_gamerounds  retention_1  retention_7
2          377  gate_40             165            1            0
3          483  gate_40               1           

In [14]:
# Retention 1

val30 = gate30['retention_1'].values
val40 = gate40['retention_1'].values

with pm.Model() as gate_retention_model:
  prior30 = pm.Beta('prior30', alpha=1, beta=1)
  prior40 = pm.Beta('prior40', alpha=1, beta=1)

  observed30 = pm.Bernoulli('observed30', p=prior30, observed=val30)
  observed40 = pm.Bernoulli('observed40', p=prior40, observed=val40)

  difference = pm.Deterministic('difference', prior40-prior30)

  trace1 = pm.sample()

az.summary(trace1, var_names=["prior30", "prior40", "difference"])

Output()

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
prior30,0.448,0.002,0.444,0.453,0.0,0.0,1854.0,1551.0,1.0
prior40,0.442,0.002,0.438,0.447,0.0,0.0,2139.0,1393.0,1.0
difference,-0.006,0.004,-0.013,0.000,0.0,0.0,2178.0,1537.0,1.0


In [15]:
# Retention 7

val30 = gate30['retention_7'].values
val40 = gate40['retention_7'].values

with pm.Model() as gate_retention_model:
  prior30 = pm.Beta('prior30', alpha=1, beta=1)
  prior40 = pm.Beta('prior40', alpha=1, beta=1)

  observed30 = pm.Bernoulli('observed30', p=prior30, observed=val30)
  observed40 = pm.Bernoulli('observed40', p=prior40, observed=val40)

  difference = pm.Deterministic('difference', prior40-prior30)

  trace7 = pm.sample()

az.summary(trace7, var_names=["prior30", "prior40", "difference"])

Output()

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
prior30,0.190,0.002,0.187,0.193,0.0,0.0,2064.0,1471.0,1.0
prior40,0.182,0.002,0.179,0.185,0.0,0.0,2093.0,1480.0,1.0
difference,-0.008,0.003,-0.013,-0.003,0.0,0.0,1977.0,1609.0,1.0


In [16]:
# R1 - decrease
# R2 - decrease
# Biggest challenge was choosing between the different options of distributions and finding out what worked and what didn't.